# LLM Detective — Solución / Guía del instructor

**Masterclass (60 min):** *Explorando los LLM por dentro*

> 📝 *Nota pedagógica:* Este cuaderno es la **versión con respuestas orientativas**. Úsalo para preparar la sesión o para dar feedback. No se lo des al alumnado como "solución única": el reto valora el razonamiento, no la respuesta exacta.

## Demos de apoyo
- Tiktokenizer: https://tiktokenizer.vercel.app/
- BBycroft LLM: https://bbycroft.net/llm

## Metodología recordada
Hipótesis → Experimento → Observación → Cambio de variable → Conclusión.

## Parte A — Tokens y contexto

### Hipótesis orientativa (lo que suelen decir los alumnos)
“Creo que el **inglés usará menos tokens** que el español porque los tokenizadores se entrenan mayormente con corpus en inglés, y las palabras en otros idiomas se parten en más sub-palabras.”

### Observación esperada (confirmada en Tiktokenizer con `cl100k_base`)
- **Español:** una palabra como `café` suele ser **1–2 tokens**; frases largas se segmentan más finamente que en inglés.
- **Inglés:** suele necesitar **~0,75 tokens por palabra** (aprox. 4 caracteres/token).
- **Código:** se segmenta en *subword* y símbolos (`def`, `_`, `(`, `)`); un bloque Python puede tener muchos tokens cortos.
- **Emoji:** cada emoji suele ser **1 token propio** (a veces más si lleva modificadores), aunque visualmente es “1 carácter”.

### ¿Por qué varía? (explicación del instructor)
El tokenizador aprende a **comprimir** el texto del corpus de entrenamiento. Donde hay más datos (inglés, código), las unidades son más largas y eficientes. En idiomas o símbolos menos frecuentes, el mismo texto ocupa **más tokens** → más coste y más latencia por token.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    texto = "El modelo procesa tokens, no palabras."
    print(len(enc.encode(texto)), "tokens")  # Ejemplo de salida: 9 tokens
except Exception as e:
    print("tiktoken no instalado; usa https://tiktokenizer.vercel.app/ en el navegador.")

# --- Ejemplo concreto para la clase ---
# 'café' en español -> a menudo 1 token (o 2 si se parte en 'c' + 'afé' según el tokenizer)
# 'def suma(a, b):' (código) -> varios tokens cortos: def | _ | sum | a | ( | , | b | ) | :
# Esto ilustra que el código NO es 'barato' en tokens: se fragmenta en muchas piezas.

## Preguntas de la Parte A — Respuestas orientativas

1. **¿Cuántos tokens?** Un texto de ~50 palabras en español puede dar ~70–90 tokens, frente a ~60–70 en inglés. El alumnado suele confirmar que el español es *algo* menos eficiente.
2. **¿Cómo se segmenta?** Sí, se ven palabras partidas a mitad (p. ej. `increí` + `ble`). Acentos y ñ suelen ir dentro del token si es frecuente; si no, se parten.
3. **Implicaciones:**
   - **Contexto:** 8k tokens ≈ unas 5–8 páginas de texto; hay que gestionar la ventana.
   - **Coste:** sí, más tokens = más caro; el idioma influye en la factura.
   - **Latencia:** más tokens de salida = más pasos de inferencia = más tiempo. Confirmado al cambiar la variable (código).

**Conclusión tipo:** “La tokenización depende del idioma y del tipo de texto; diseñar prompts cortos y en el idioma mejor soportado ahorra coste y latencia.”

## Parte B — Decisión de ingeniería (respuesta orientativa)

### El caso
Asistente documental con información privada, presupuesto limitado, buena latencia y posible infraestructura propia.

### Criterios de evaluación sugeridos (el instructor debe valorar que el alumno los *justifique*)
1. **Privacidad / cumplimiento:** los datos de clientes no pueden salir a terceros sin garantías.
2. **Coste:** precio por token o por hora de GPU, y si escala con el uso.
3. **Latencia:** tiempo de primera respuesta (TTFT) y tokens/seg.
4. **Calidad:** acierto en tareas documentales (RAG / extracción).
5. **Control / mantenimiento:** poder afinar, auditar y versionar el modelo.

### Familias/modelos a investigar (sin obligar a uno concreto)
- **Open-weight autohospedado** (p. ej. familias tipo Llama, Mistral, Qwen): máxima privacidad y control, CAPEX de GPU, latencia predecible en local.
- **API gestionada con DPA / región UE:** privacidad contractual, OPEX por uso, sin mantener infraestructura.
- **Modelo pequeño + RAG** suele bastar para “asistente documental”, priorizando latencia y coste.

### Experimento / benchmark propuesto
- Mismo conjunto de 50 consultas reales contra **Opción A (open-weight local)** y **Opción B (API con DPA)**.
- Métricas: coste total, p95 latencia, y `groundedness`/precisión con respuestas revisadas por humano.
- Decisión basada en: si el volumen es alto y la privacidad crítica → infraestructura propia; si el volumen es variable → API con DPA.

**Conclusión orientativa:** priorizar **open-weight en infraestructura propia o API con acuerdo de privacidad (DPA)**, modelo mediano con RAG, y medir coste/latencia antes de escalar.

## Cierre

**Pregunta de transferencia:** *“Si un modelo ‘habla’ en tokens y no en palabras, ¿cómo cambia la forma en que diseñas un producto con LLM (prompts, costes, privacidad, idioma)?”*

Puntos a resaltar con el alumnado:
- El **idioma y el tipo de texto** afectan directamente al coste y la latencia.
- La **privacidad** puede decidir entre open-weight propio y API con DPA.
- Diseñar con LLM es también **ingeniería de tokens**, no solo de prompts.

Demos:
- Tiktokenizer: https://tiktokenizer.vercel.app/
- BBycroft LLM: https://bbycroft.net/llm